# MCP (Model Context Protocol) Server Implementation

In [6]:
!python -m pip install fastmcp langchain_mcp_adapters --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
os.makedirs("mcpserver",exist_ok=True)

In [8]:
%%writefile mcpserver/mcpserver1.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
async def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
async def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='streamable-http')


Overwriting mcpserver/mcpserver1.py


In [9]:
# run mCP server: python mcpserver/mcpserver1.py

## Implement an Agent which connects to the MCP server and fetches the tools

In [10]:
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [11]:
client = MultiServerMCPClient({"TredenceMCP":{"url":"http://127.0.0.1:8000/mcp",
                                              "transport":"streamable_http"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description=' this funciton can be used to get current weather information', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7764028723e0>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x776402928b80>)]

In [12]:
agent = create_react_agent("google_genai:gemini-2.0-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='1fc5dd94-367c-4683-b28c-a105240fd207'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--ba7e88e3-69b3-48d1-a483-1f671dd4875b-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': 'a830b819-f40d-46f1-9736-584fb8186350', 'type': 'tool_call'}], usage_metadata={'input_tokens': 36, 'output_tokens': 7, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='{\n  "city": "Delhi",\n  "weather": "overcast clouds",\n  "temperature": 306,\n  "unit": "kelvin"\n}', name='get_current_weather', id='cc15eed7-a78e-4362-849b-3d39ac2d063b', tool_call_id='a830b819-f40d-46f1-9736-584fb818

In [13]:
await agent.ainvoke({"messages":[{"role":'user',"content":"Tell me more about city Jaiselmer?"}]})

{'messages': [HumanMessage(content='Tell me more about city Jaiselmer?', additional_kwargs={}, response_metadata={}, id='84eb4aa7-eb3e-4aab-b246-bbbf5ae4da0c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_wikipedia_summary', 'arguments': '{"query": "Jaiselmer"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--caab6696-c73d-446f-acd9-1e44425bffe2-0', tool_calls=[{'name': 'get_wikipedia_summary', 'args': {'query': 'Jaiselmer'}, 'id': 'd8e1eeee-7702-4fb2-aef8-00abd34e7fa7', 'type': 'tool_call'}], usage_metadata={'input_tokens': 38, 'output_tokens': 9, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="Jaisalmer , nicknamed The Golden city, is a city in the north-western Indian state of Rajasthan, located 575 kilometres (357 mi) west of the state capital Jaipur, in the heart of the Thar Desert. It se

In [14]:
%%writefile mcpserver/mcpserver2.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
async def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
async def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='stdio')


Writing mcpserver/mcpserver2.py


In [17]:
client = MultiServerMCPClient({"TredenceMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                              "transport":"stdio"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description=' this funciton can be used to get current weather information', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7763f9607ba0>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7763eafa4a40>)]

In [18]:
agent = create_react_agent("google_genai:gemini-2.0-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='0056e2d2-8139-4a1f-b798-6563df70251e'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--2da65343-267a-4697-9be9-a8a8f55c821b-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': '3c4154c0-5421-4ae6-85a8-f15856ff6a4f', 'type': 'tool_call'}], usage_metadata={'input_tokens': 36, 'output_tokens': 7, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='{\n  "city": "Delhi",\n  "weather": "overcast clouds",\n  "temperature": 306,\n  "unit": "kelvin"\n}', name='get_current_weather', id='e2276832-ce5c-48ea-a04f-617dabf9d426', tool_call_id='3c4154c0-5421-4ae6-85a8-f15856ff